In [7]:
class AirConditioner:
    """A room air-conditioner. Reported buggy by the QA team - fix it!"""
    VALID_MODES = ("cool", "fan", "dry", "auto")
    MIN_TEMP = 16
    MAX_TEMP = 30

    def __init__(self, brand, room_name, temperature=25, mode="cool", fan_speed=1):
        self.brand = brand
        self.room_name = room_name
        self.is_on = False

        # REASON (Bug 5): Using 'self.temperature' routes through the setter validation.
        # Direct assignment to 'self._temperature' was bypassing the range check during object creation.
        self.temperature = temperature

        self.mode = mode
        self.fan_speed = fan_speed

        # REASON (Bug 3): Removed 'self._is_energy_saving' from __init__.
        # Storing a snapshot here caused the status to remain static instead of updating dynamically.

    @property
    def temperature(self):
        return self._temperature

    @temperature.setter
    def temperature(self, value):
        # REASON (Bug 4): Changed logical 'and' to 'or'.
        # A single number cannot be both < 16 and > 30 at the same time, which allowed bad values to pass.
        if value < self.MIN_TEMP or value > self.MAX_TEMP:
            raise ValueError(f"Temperature must be {self.MIN_TEMP}-{self.MAX_TEMP} C.")

        # REASON (Bug 2): Must store into private '_temperature'.
        # Assigning to 'self.temperature' recursively invoked this setter forever, causing a RecursionError.
        self._temperature = value

    @property
    def mode(self):
        return self._mode

    @mode.setter
    def mode(self, value):
        if value not in self.VALID_MODES:
            raise ValueError(f"Mode must be one of {self.VALID_MODES}.")
        self._mode = value

    @property
    def fan_speed(self):
        return self._fan_speed

    @fan_speed.setter
    def fan_speed(self, value):
        if value not in (1, 2, 3):
            raise ValueError("Fan speed must be 1 (low), 2 (medium) or 3 (high).")
        self._fan_speed = value

    @property
    def is_energy_saving(self):
        # REASON (Bug 3): Calculate dynamically on each read using the current temperature
        # to ensure it immediately reflects changes made after object initialization.
        return self._temperature >= 25

    def turn_on(self):
        self.is_on = True

    def turn_off(self):
        self.is_on = False

    def cooler(self):
        # REASON (Bug 6): Added lower-bound guard.
        # Decrementing without checking MIN_TEMP allowed the temperature to drop below 16°C.
        if self._temperature > self.MIN_TEMP:
            self._temperature -= 1

    def warmer(self):
        if self._temperature < self.MAX_TEMP:
            self._temperature += 1

    def __str__(self):
        power = "ON" if self.is_on else "OFF"
        # REASON (Bug 1): Replaced 'self.fan' with 'self.fan_speed'.
        # The attribute defined in __init__ is fan_speed; referencing self.fan raised an AttributeError.
        return (f"{self.brand} AC in {self.room_name}: {power}, "
                f"{self.temperature}C, mode={self.mode}, fan={self.fan_speed}")